# Vigil — Notebook 05: Simulador de Escenarios y Reporte de Inteligencia

Este cuaderno calcula el índice final de PDIV (Posicionamiento Digital de Intención de Voto), ejecuta simulaciones de escenarios temáticos ('What-If') y compila el reporte de auditoría semanal.

In [ ]:
import polars as pl
import yaml
import sys
import os
sys.path.append('..')
from src.analysis.simulator import WhatIfSimulator

## 1. Cargar Datos y Calcular PDIV Base (Baseline)

In [ ]:
posts_gold = pl.read_parquet("../data/gold/redes_sociales/fb_posts_gold.parquet")
comments_gold = pl.read_parquet("../data/gold/redes_sociales/fb_comments_gold.parquet")

# Combinar datasets
df_gold = pl.concat([
    posts_gold.select(["candidato", "sentimiento", "reacciones_totales", "texto_publicacion", "keywords_extracted"]),
    comments_gold.select(["candidato", "sentimiento", "reacciones_totales", "texto_publicacion", "keywords_extracted"])
])

# Cargar pesos y config
with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

# Inicializar Simulador
simulator = WhatIfSimulator()

# Calcular scores de línea base
df_base = simulator.calcular_scores_base(df_gold)
print("RESULTADOS BASELINE PDIV ELECTORAL:")
print(df_base.select(["candidato", "pdiv_score", "sentimiento_score", "volumen_score", "engagement_score"]))

## 2. Simulación de Escenario Temático ('What-If')

In [ ]:
# Simularemos el escenario: ¿Qué pasa si Beatriz Estrada incrementa un 40%
# sus publicaciones asociadas a la palabra 'amor' (temas sociales del DIF)?
ajustes = {"amor": 0.40, "cercanía": 0.30}

df_sim = simulator.simular_cambio_temas(df_gold, "Beatriz Estrada", ajustes)

print("RESULTADO DE SIMULACIÓN DE ESCENARIO:")
print(df_sim)

## 3. Generar Reporte de Inteligencia Electoral Semanal (.md)

In [ ]:
os.makedirs("../reports", exist_ok=True)
reporte_path = "../reports/reporte_semanal_2026-06-01.md"

with open(reporte_path, "w", encoding="utf-8") as f:
    f.write("# REPORT DE INTELIGENCIA SEMANAL: TEPIC 2026\n")
    f.write("Fecha de reporte: 2026-06-01 | Capa de datos: Gold\n\n")
    
    f.write("## 1. Posicionamiento Digital de Intención de Voto (PDIV)\n")
    f.write("El índice PDIV sintetiza volumen, sentimiento neto y engagement normalizado:\n\n")
    
    # Escribir tabla base
    f.write("| Candidato | PDIV Score | Sentimiento Score | Volumen Score | Engagement Score |\n")
    f.write("| :--- | :---: | :---: | :---: | :---: |\n")
    for row in df_base.iter_rows(named=True):
        f.write(f"| **{row['candidato']}** | {round(row['pdiv_score'], 2)} | {round(row['sentimiento_score'], 2)} | {round(row['volumen_score'], 2)} | {round(row['engagement_score'], 2)} |\n")
        
    f.write("\n## 2. Recomendaciones de Escenario (Simulado 'What-If')\n")
    f.write("Análisis de sensibilidad aplicando +40% de peso a tópicos de 'Gestión Social / Amor' para Beatriz Estrada:\n\n")
    
    f.write("| Candidato | PDIV Base | PDIV Simulado | Diferencia | Sentimiento Simulado |\n")
    f.write("| :--- | :---: | :---: | :---: | :---: |\n")
    for row in df_sim.iter_rows(named=True):
        f.write(f"| **{row['candidato']}** | {round(row['pdiv_score'], 2)} | {round(row['pdiv_score_sim'], 2)} | **{row['dif_pdiv']}** | {round(row['sentimiento_score_sim'], 2)} |\n")
        
    f.write("\n\n*Informe técnico generado automáticamente por el Framework de Inteligencia Electoral Local (FIEL) — RndmStudio 2026.*")

print(f"Reporte de Inteligencia escrito con éxito en: {reporte_path}")